In [2]:
!pip install -q --upgrade qiskit qiskit-aer qiskit-ibm-runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 87.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 88.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.0/218.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 7.6 MB/s eta 0:00:00


In [14]:
# ── Block 1: Imports & Constants ─────────────────────────────────────────
import numpy as np
import random
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from kaggle_secrets import UserSecretsClient

NQUBITS   = 2
SHOTS     = 1024
TOLERANCE = 2.0   # higher = less sensitive, lower = more sensitive

In [15]:
# ── Block 2: Connect to IBM Backend ──────────────────────────────────────
secrets = UserSecretsClient()
token   = secrets.get_secret("IBM_QUANTUM_TOKEN")

service = QiskitRuntimeService(
    token    = token,
    instance = "open-instance"
)

backend = service.least_busy(simulator=False, operational=True, min_num_qubits=2)
pm      = generate_preset_pass_manager(backend=backend, optimization_level=1,
                                        translation_method="translator")

print(f"Backend : {backend.name}")
print(f"Qubits  : {backend.num_qubits}")
print(f"Pending : {backend.status().pending_jobs}")

qiskit_runtime_service._discover_account:WARNING:2026-06-02 14:13:14,898: Loading account with the given token. A saved account will not be used.


Backend : ibm_marrakesh
Qubits  : 156
Pending : 0


In [16]:
# ── Block 3: Build 2-Qubit Grover Circuits (correct + buggy) ─────────────
def build_grover_2q(inject_x_on=None):
    qc = QuantumCircuit(NQUBITS, NQUBITS)
    qc.h([0, 1])
    qc.cz(0, 1)
    qc.h([0, 1])
    qc.x([0, 1])
    qc.h(1)
    qc.cx(0, 1)
    qc.h(1)
    qc.x([0, 1])
    qc.h([0, 1])
    if inject_x_on is not None:
        qc.x(inject_x_on)
    qc.measure([0, 1], [0, 1])
    return qc

bug_qubit      = random.choice([0, 1])
grover_correct = build_grover_2q()
grover_buggy   = build_grover_2q(inject_x_on=bug_qubit)

print("Correct circuit:")
print(grover_correct.draw(output="text", fold=-1))
print(f"\nBuggy circuit (X injected on q{bug_qubit}):")
print(grover_buggy.draw(output="text", fold=-1))

Correct circuit:
     ┌───┐   ┌───┐┌───┐          ┌───┐┌───┐     ┌─┐   
q_0: ┤ H ├─■─┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────┤M├───
     ├───┤ │ ├───┤├───┤┌───┐┌─┴─┐├───┤├───┤┌───┐└╥┘┌─┐
q_1: ┤ H ├─■─┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├─╫─┤M├
     └───┘   └───┘└───┘└───┘└───┘└───┘└───┘└───┘ ║ └╥┘
c: 2/════════════════════════════════════════════╩══╩═
                                                 0  1 

Buggy circuit (X injected on q0):
     ┌───┐   ┌───┐┌───┐          ┌───┐┌───┐┌───┐┌─┐   
q_0: ┤ H ├─■─┤ H ├┤ X ├───────■──┤ X ├┤ H ├┤ X ├┤M├───
     ├───┤ │ ├───┤├───┤┌───┐┌─┴─┐├───┤├───┤├───┤└╥┘┌─┐
q_1: ┤ H ├─■─┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├─╫─┤M├
     └───┘   └───┘└───┘└───┘└───┘└───┘└───┘└───┘ ║ └╥┘
c: 2/════════════════════════════════════════════╩══╩═
                                                 0  1 


In [17]:
# ── Block 4: Ideal Distribution (Noiseless Simulator) ────────────────────
def counts_to_probs(counts, nqubits):
    total = sum(counts.values())
    probs = np.zeros(2 ** nqubits)
    for bitstring, count in counts.items():
        probs[int(bitstring, 2)] = count / total
    return probs

simulator_ideal = AerSimulator()
t_ideal         = transpile(grover_correct, simulator_ideal)
counts_ideal    = simulator_ideal.run(t_ideal, shots=SHOTS).result().get_counts()
probs_ideal     = counts_to_probs(counts_ideal, NQUBITS)

print("Ideal distribution:")
for i, p in enumerate(probs_ideal):
    if p > 1e-4:
        print(f"  |{format(i, f'0{NQUBITS}b')}>  {p:.4f}")

Ideal distribution:
  |11>  1.0000


In [18]:
# ── Block 5: Per-Qubit Combined Error from IBM Calibration Data ───────────
# gate_err[q]  = 1 - prod(1 - gate_error_i)  for all gates touching q
# readout[q]   = measurement error
# combined[q]  = 1 - (1 - gate_err[q]) * (1 - readout[q])

def get_per_qubit_combined_error(backend):
    props  = backend.properties()
    nq     = backend.num_qubits

    readout = {}
    for q, qprops in enumerate(props.qubits):
        ro = next((float(p.value) for p in qprops if p.name == "readout_error"), None)
        if ro is None:
            m01 = next((float(p.value) for p in qprops if p.name == "prob_meas0_prep1"), None)
            m10 = next((float(p.value) for p in qprops if p.name == "prob_meas1_prep0"), None)
            ro  = (m01 + m10) / 2.0 if (m01 and m10) else 0.0
        readout[q] = ro

    log_fid = {q: 0.0 for q in range(nq)}
    for gate in props.gates:
        err = next((float(p.value) for p in gate.parameters if p.name == "gate_error"), None)
        if err is None:
            continue
        for q in gate.qubits:
            if q in log_fid:
                log_fid[q] += np.log(max(1.0 - err, 1e-12))

    gate_err = {q: 1.0 - np.exp(log_fid[q]) for q in range(nq)}
    combined = {q: 1.0 - (1.0 - gate_err[q]) * (1.0 - readout[q]) for q in range(nq)}
    return gate_err, readout, combined

gate_err, readout_err, combined_err = get_per_qubit_combined_error(backend)

t_correct    = pm.run(grover_correct)
used_qubits  = sorted({t_correct.find_bit(q).index for q in t_correct.qubits})
virt_to_phys = {v: used_qubits[v] for v in range(NQUBITS)}

print(f"Physical qubits used: {used_qubits}")
print(f"\n{'Qubit':>8}  {'Gate Err':>10}  {'Readout Err':>12}  {'Combined Err':>13}")
print("-" * 50)
for v in range(NQUBITS):
    pq = virt_to_phys[v]
    print(f"  q{v}→q{pq:<3}  {gate_err[pq]:>10.6f}  {readout_err[pq]:>12.6f}  {combined_err[pq]:>13.6f}")

Physical qubits used: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155]

   Qubit    Gate Err   Readout Err   Combined Err
--------------------------------------------------
  q0→q0      0.024422      0.010498       0.034664
  q1→q1      0.030994      0.004761       0.035607


In [19]:
# ── Block 6: Run Both Circuits on Real IBM Hardware ───────────────────────
def sampler_to_counts(result_entry):
    try:
        return result_entry.data.c.get_counts()
    except AttributeError:
        pass
    try:
        reg = list(result_entry.data.keys())[0]
        return result_entry.data[reg].get_counts()
    except Exception:
        pass
    raise RuntimeError("Cannot extract counts from SamplerV2 result.")

t_correct_hw = pm.run(grover_correct)
t_buggy_hw   = pm.run(grover_buggy)

sampler = Sampler(mode=backend)
job     = sampler.run([t_correct_hw, t_buggy_hw], shots=SHOTS)
result  = job.result()

counts_hw_correct = sampler_to_counts(result[0])
counts_hw_buggy   = sampler_to_counts(result[1])
probs_hw_correct  = counts_to_probs(counts_hw_correct, NQUBITS)
probs_hw_buggy    = counts_to_probs(counts_hw_buggy,   NQUBITS)

print("Hardware distribution — correct circuit:")
for i, p in enumerate(probs_hw_correct):
    if p > 1e-4:
        print(f"  |{format(i, f'0{NQUBITS}b')}>  {p:.4f}")

print("\nHardware distribution — buggy circuit:")
for i, p in enumerate(probs_hw_buggy):
    if p > 1e-4:
        print(f"  |{format(i, f'0{NQUBITS}b')}>  {p:.4f}")

Hardware distribution — correct circuit:
  |00>  0.0020
  |01>  0.0127
  |10>  0.0195
  |11>  0.9658

Hardware distribution — buggy circuit:
  |00>  0.0078
  |01>  0.0010
  |10>  0.9824
  |11>  0.0088


In [20]:
# ── Block 7: Per-Qubit Observed Flip Rate ────────────────────────────────
# flip_rate[q] = |P_hw(q=0) - P_ideal(q=0)|
# Qiskit MSB-first: qubit q is at index (nqubits-1-q) in bitstring

def marginal_p0(probs, nqubits, q):
    return sum(
        probs[i]
        for i in range(2 ** nqubits)
        if format(i, f"0{nqubits}b")[nqubits - 1 - q] == "0"
    )

flip_correct = {}
flip_buggy   = {}
for q in range(NQUBITS):
    p0i = marginal_p0(probs_ideal,      NQUBITS, q)
    p0c = marginal_p0(probs_hw_correct, NQUBITS, q)
    p0b = marginal_p0(probs_hw_buggy,   NQUBITS, q)
    flip_correct[q] = abs(p0c - p0i)
    flip_buggy[q]   = abs(p0b - p0i)

print(f"\n{'Q':>4}  {'P0 ideal':>10}  {'P0 hw_cor':>10}  {'flip(cor)':>10}  {'P0 hw_bug':>10}  {'flip(bug)':>10}")
print("-" * 65)
for q in range(NQUBITS):
    p0i = marginal_p0(probs_ideal,      NQUBITS, q)
    p0c = marginal_p0(probs_hw_correct, NQUBITS, q)
    p0b = marginal_p0(probs_hw_buggy,   NQUBITS, q)
    print(f"  q{q}  {p0i:>10.4f}  {p0c:>10.4f}  {flip_correct[q]:>10.4f}  {p0b:>10.4f}  {flip_buggy[q]:>10.4f}")


   Q    P0 ideal   P0 hw_cor   flip(cor)   P0 hw_bug   flip(bug)
-----------------------------------------------------------------
  q0      0.0000      0.0215      0.0215      0.9902      0.9902
  q1      0.0000      0.0146      0.0146      0.0088      0.0088


In [21]:
# ── Block 8: Correlation — IBM Combined Error vs Observed Flip Rate ───────
hw_mean   = np.mean([combined_err[virt_to_phys[q]] for q in range(NQUBITS)])
flip_mean = np.mean([flip_correct[q] for q in range(NQUBITS)])

print("\nIBM Combined Error  vs  Observed Flip Rate (correct circuit)")
print("=" * 62)
print(f"  {'VQ':>2}  {'PQ':>3}  {'IBM Combined':>13}  {'Flip Rate':>10}  Consistent?")
print("-" * 62)
for v in range(NQUBITS):
    pq = virt_to_phys[v]
    ce = combined_err[pq]
    fr = flip_correct[v]
    ok = (ce < hw_mean) == (fr < flip_mean)
    print(f"  q{v}  q{pq:<2}  {ce:>13.6f}  {fr:>10.6f}  {'yes' if ok else 'no'}")
print("=" * 62)

safest   = min(range(NQUBITS), key=lambda q: combined_err[virt_to_phys[q]])
riskiest = max(range(NQUBITS), key=lambda q: combined_err[virt_to_phys[q]])
print(f"\n  Safest   → q{safest}  combined={combined_err[virt_to_phys[safest]]:.6f}  flip={flip_correct[safest]:.6f}")
print(f"  Riskiest → q{riskiest}  combined={combined_err[virt_to_phys[riskiest]]:.6f}  flip={flip_correct[riskiest]:.6f}")


IBM Combined Error  vs  Observed Flip Rate (correct circuit)
  VQ   PQ   IBM Combined   Flip Rate  Consistent?
--------------------------------------------------------------
  q0  q0        0.034664    0.021484  no
  q1  q1        0.035607    0.014648  no

  Safest   → q0  combined=0.034664  flip=0.021484
  Riskiest → q1  combined=0.035607  flip=0.014648


In [22]:
# ── Block 9: Per-Qubit Bug Detection via Threshold ────────────────────────
# threshold[q] = combined_err[q] * TOLERANCE
# flip_buggy[q] > threshold[q]  →  BUG
# flip_buggy[q] <= threshold[q] →  hardware noise
#
# Tune TOLERANCE in Block 1:
#   2.0 = default | 1.5 = stricter | 3.0 = looser

print(f"\nPer-Qubit Bug Detection  (TOLERANCE = {TOLERANCE}x)")
print("=" * 68)
print(f"  {'Q':>2}  {'Combined Err':>13}  {'Threshold':>10}  {'flip(bug)':>10}  Verdict")
print("-" * 68)

any_bug = False
for q in range(NQUBITS):
    pq        = virt_to_phys[q]
    ce        = combined_err[pq]
    threshold = ce * TOLERANCE
    fb        = flip_buggy[q]
    bug       = fb > threshold
    if bug:
        any_bug = True
    tag = "  <- BUG QUBIT" if q == bug_qubit else ""
    print(f"  q{q}  {ce:>13.6f}  {threshold:>10.6f}  {fb:>10.6f}  {'BUG' if bug else 'noise'}{tag}")

print("=" * 68)
print(f"\n  threshold[q] = combined_err[q] x {TOLERANCE}")
print(f"  OVERALL VERDICT: {'BUG DETECTED' if any_bug else 'NO BUG — within hardware noise'}")


Per-Qubit Bug Detection  (TOLERANCE = 2.0x)
   Q   Combined Err   Threshold   flip(bug)  Verdict
--------------------------------------------------------------------
  q0       0.034664    0.069328    0.990234  BUG  <- BUG QUBIT
  q1       0.035607    0.071214    0.008789  noise

  threshold[q] = combined_err[q] x 2.0
  OVERALL VERDICT: BUG DETECTED
